# Qwen3-14B GRPO


In [ ]:
import os, subprocess, sys
from pathlib import Path

LAUNCH_DIR = Path.cwd().resolve()
subprocess.run([sys.executable, "-m", "pip", "install", "python-dotenv>=1,<2"], check=True)
from dotenv import load_dotenv

ENV_FILE = Path(os.environ.get("CRASHDIAG_ENV_FILE", LAUNCH_DIR / ".env")).expanduser()
if not ENV_FILE.is_absolute():
    ENV_FILE = (LAUNCH_DIR / ENV_FILE).resolve()
if not ENV_FILE.is_file():
    raise RuntimeError(f"CrashDiag .env not found: {ENV_FILE}")
load_dotenv(ENV_FILE, override=True)
os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")

REPO_URL = os.environ.get("CRASHDIAG_REPO_URL", "https://github.com/Indium-AI-Labs/CrashDiag.git")
SOURCE_COMMIT = os.environ.get("CRASHDIAG_SOURCE_COMMIT", "main")
WORKDIR = Path(os.environ.get("CRASHDIAG_WORKDIR", LAUNCH_DIR / "CrashDiag-runtime")).expanduser().resolve()
if (WORKDIR / ".git").is_dir():
    subprocess.run(["git", "-C", str(WORKDIR), "fetch", "origin", "main"], check=True)
elif WORKDIR.exists() and any(WORKDIR.iterdir()):
    raise RuntimeError(f"CRASHDIAG_WORKDIR exists and is not a Git checkout: {WORKDIR}")
else:
    subprocess.run(["git", "clone", REPO_URL, str(WORKDIR)], check=True)
subprocess.run(["git", "-C", str(WORKDIR), "checkout", SOURCE_COMMIT], check=True)
os.chdir(WORKDIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-U", "bitsandbytes"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[train]"], check=True)
print(f"env_file={ENV_FILE}")
print("checked_out_source_commit=" + subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())


In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo
import os

BASE_MODEL = "Qwen/Qwen3-14B"
MODEL_SLUG = "qwen3_14b"
BUCKET_ID = "devaanshpa/CrashDiag"
DATASET_RUN_ID = os.environ.get("CRASHDIAG_DATASET_RUN_ID", "").strip()
def ist_run_id(stage):
    return datetime.now(ZoneInfo("Asia/Kolkata")).strftime("%Y%m%dT%H%M%SIST") + f"-{MODEL_SLUG}-{stage}"
if not DATASET_RUN_ID:
    raise RuntimeError("Set CRASHDIAG_DATASET_RUN_ID to the fresh dataset-generation run ID.")
print(f"base_model={BASE_MODEL}")
print(f"dataset_run_id={DATASET_RUN_ID}")


In [ ]:
from pathlib import Path
from training.artifacts import ArtifactConfig, ArtifactUploader

SFT_RUN_ID = os.environ.get("CRASHDIAG_SFT_RUN_ID", "").strip()
if not SFT_RUN_ID: raise RuntimeError("Set CRASHDIAG_SFT_RUN_ID to the completed SFT run ID.")
GRPO_RUN_ID = os.environ.get("CRASHDIAG_GRPO_RUN_ID") or ist_run_id("grpo")
CURRICULUM = os.environ.get("CRASHDIAG_CURRICULUM", "hard-v3").strip().lower()
TRAIN_FILE = "grpo_hard_train.jsonl" if CURRICULUM == "hard-v3" else "grpo_train.jsonl"
EVAL_FILE = "grpo_hard_eval.jsonl" if CURRICULUM == "hard-v3" else "grpo_eval.jsonl"
DATASET_DIR, SFT_DIR = Path("artifacts/datasets"), Path("artifacts/sft")
ArtifactUploader(ArtifactConfig(bucket_id=BUCKET_ID, run_id=DATASET_RUN_ID, token=os.environ["HF_TOKEN"])).download_stage("datasets", DATASET_DIR)
ArtifactUploader(ArtifactConfig(bucket_id=BUCKET_ID, run_id=SFT_RUN_ID, token=os.environ["HF_TOKEN"])).download_stage("sft", SFT_DIR)
assert (DATASET_DIR / TRAIN_FILE).is_file(), f"dataset stage missing {TRAIN_FILE}; check CRASHDIAG_DATASET_RUN_ID={DATASET_RUN_ID}"
assert (DATASET_DIR / EVAL_FILE).is_file(), f"dataset stage missing {EVAL_FILE}; check CRASHDIAG_DATASET_RUN_ID={DATASET_RUN_ID}"
print(f"GRPO_RUN_ID={GRPO_RUN_ID}")
print(f"curriculum={CURRICULUM}")
print(f"train_file={TRAIN_FILE}")
print(f"eval_file={EVAL_FILE}")


In [ ]:
import subprocess, sys

common = [
    sys.executable, "-m", "accelerate.commands.launch",
    "--num_processes", "1", "--num_machines", "1",
    "--mixed_precision", "bf16", "--dynamo_backend", "no",
    "-m", "training.grpo", "--model", str(SFT_DIR),
    "--train-file", str(DATASET_DIR / TRAIN_FILE),
    "--eval-file", str(DATASET_DIR / EVAL_FILE),
    "--output-dir", "outputs/grpo", "--load-in-4bit", "--precision", "bf16",
    "--batch-size", "2", "--gradient-accumulation-steps", "4", "--num-generations", "2",
    "--max-prompt-length", "1024", "--max-completion-length", "64",
    "--max-steps", "24", "--artifact-bucket", BUCKET_ID, "--run-id", GRPO_RUN_ID,
    "--artifact-stage", "grpo-smoke", "--sandbox-url", os.environ["CRASHDIAG_SANDBOX_URL"],
]
subprocess.run(common, check=True)
full = common[:]
full[full.index("24")] = "96"
full[full.index("grpo-smoke")] = "grpo"
subprocess.run(full, check=True)


In [ ]:
from training.evaluate_jsonl import main as evaluate_main

exit_code = evaluate_main([
    "--model", "outputs/grpo", "--dataset", str(DATASET_DIR / EVAL_FILE),
    "--output-dir", "outputs/grpo-eval", "--load-in-4bit", "--precision", "bf16",
    "--max-new-tokens", "64",
    "--sandbox-url", os.environ["CRASHDIAG_SANDBOX_URL"],
    "--artifact-bucket", BUCKET_ID, "--run-id", GRPO_RUN_ID, "--artifact-stage", "grpo-eval",
])
if exit_code: raise RuntimeError(f"GRPO evaluation failed: {exit_code}")


In [ ]:
from IPython.display import SVG, display

REPORTS_DIR = Path("outputs/grpo-eval") / "reports"
charts = sorted(REPORTS_DIR.glob("*.svg"))
if not charts:
    raise RuntimeError(f"No evaluation SVG charts were generated in {REPORTS_DIR}")
print(f"Uploaded evaluation reports: hf://buckets/{BUCKET_ID}/runs/{GRPO_RUN_ID}/grpo-eval/reports")
for chart in charts:
    display(SVG(filename=str(chart)))
